## 0. 準備環境（直接執行，不必逐行看懂）
先執行下面的安裝格，等待時可以聽老師介紹 GMT／PyGMT，或閱讀後面的課程導覽。這兩格負責安裝套件，不必逐行看懂。

**Colab 請分開執行。** 第一格安裝 Conda 後可能自動重啟；等重新連線，再執行第二格。勿在安裝期間重複按執行。

本機已裝好環境時會跳過安裝。此教材使用 PyGMT 0.17 / GMT 6.5。


In [ ]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.13"])
    import condacolab
    condacolab.install()
else:
    print("本機模式：使用目前 Python 環境。")

In [ ]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if IN_COLAB:
    subprocess.check_call([
        "mamba", "install", "-y", "-c", "conda-forge",
        "pygmt=0.17", "gmt=6.5", "ghostscript=10.04", "pandas", "pillow", "ipywidgets", "ipyleaflet", "pyproj"
    ])
else:
    print("跳過 Colab 安裝。")


# 用 PyGMT 畫自己的地圖

台灣的地震都發生在哪裡？山脈往海底延伸，又會是什麼模樣？這堂課，我們用幾行 Python，把資料變成看得見的地圖。

從一個空白外框開始，親手加上海岸線、地震與地形，再換個角度看立體的台灣。先試著自己改指令，看懂每一步如何改變畫面；最後再邀請 AI 幫忙，把地圖帶到你想探索的世界角落。

不必一開始就看懂所有程式。先跑出一張圖、改一個地方，看看會發生什麼。

### 開始動手

- 在 Colab 先另存自己的副本，再由上往下執行。
- 第 1–4 主題先親手操作、不用 AI；第 5 主題再使用 Codex 或 Colab Gemini 延伸自己的想法。

想先看看這些工具能畫出什麼？從 [GMT、PyGMT 與地圖作品導覽](https://github.com/jimmy60504/pygmt-map-lab/blob/main/intro.md) 開始，等環境安裝完成後，再往下畫出第一張圖。

### 常用快捷鍵

| 操作 | Windows／Linux | Mac |
| --- | --- | --- |
| 執行目前儲存格並移到下一格 | Shift + Enter | Shift + Enter |
| 取消／切換註解 | Ctrl + / | ⌘ + / |

把游標放在程式行，或選取多行，再按切換註解的快捷鍵，即可移除或加上行首的 `#`。只選程式行，不要連中文說明一起取消註解。

修改後按 **Shift + Enter** 看結果，等執行完成再繼續下一步。


## 1. 從空白底圖，一步一步畫出台灣
先執行下一格，看到最簡單的地圖外框。接著依序移除步驟 1–5 程式行開頭的 `#`，每次只開啟一步，保留前面已開啟的步驟，再重跑整格觀察差異。

- `fig = pygmt.Figure()`：建立一張新圖；每次重跑都從頭畫，不會累積上次的內容。
- `region`：繪圖範圍，順序是 **西、東、南、北**；`projection="M15c"`：麥卡托投影、圖寬 15 公分。
- `fig.basemap()` 畫框線、刻度等；`fig.coast()` 畫海陸與海岸線；最後 `fig.show()` 顯示結果。

**先記住繪圖順序**：在同一個 `fig` 上，後畫的內容可能蓋住前面的內容。因此先填海陸顏色，再加線條、格線與標題。

後面的呼叫省略 `region` 與 `projection`，是沿用這張圖已設定的範圍與投影，**不是讀取上一層的圖片**。圖層疊加與設定沿用是兩件事。

### 第一張圖的 API 參考
不用整頁讀完：想改哪個效果，就點對應指令，在 **Parameters** 找參數，再看 **Examples**。以下連結對應課堂使用的 PyGMT 0.17。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [pygmt.Figure()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.html) | 建立一張圖 | Methods：還能加哪些內容 |
| [fig.basemap()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.basemap.html) | 底圖、刻度、格線與標題 | `region`、`projection`、`frame` |
| [fig.coast()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.coast.html) | 海陸填色與海岸線 | `land`、`water`、`shorelines`、`resolution` |
| [fig.show()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.show.html) | 顯示目前的圖 | `width`、`dpi` |

`import pygmt` 是載入套件，不是繪圖指令。`region=[119, 123, 21, 26]` 依序指定西界、東界、南界、北界。

**試著發現一個新選項**：打開 `coast` 文件，找找 `borders` 或 `rivers` 能做什麼。


In [ ]:
import pygmt

fig = pygmt.Figure()
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame="0")

# 1. 加上經緯度數字與刻度（a：標示數字；f：刻度）
# fig.basemap(frame="af")

# 2. 填上陸地與海洋顏色
# fig.coast(land="gray90", water="lightblue", resolution="h")

# 3. 加上海岸線
# fig.coast(shorelines="0.6p,gray30", resolution="h")

# 4. 加上每 1 度的經緯格線，並重畫刻度（g：格線）
# fig.basemap(frame="a1f0.5g1")

# 5. 加上標題
# fig.basemap(frame="+tTaiwan: coastlines")

fig.show()
# 圖片顯示後，可按右鍵另存圖片。


## 2. 地震在哪裡？

### 先挑一個你想畫的地震事件

不知道要選哪個事件？台灣事件可以先到氣象署看看：

- [最近地震](https://www.cwa.gov.tw/V8/C/E/index.html)：從近期有感地震找題目，點進報告看發生時間、位置與規模。
- [歷史災害地震](https://scweb.cwa.gov.tw/zh-tw/page/disaster)：認識集集、美濃等重大事件，找一個你想進一步觀察的地震。較早期的歷史記載不一定有完整的 USGS 資料，練習可先選近代事件。

想畫世界其他地方，則可以從 USGS 找題目：

- [Latest Earthquakes｜最近地震地圖](https://earthquake.usgs.gov/earthquakes/map/)：瀏覽近期全球地震，可選「30 Days, Significant Worldwide」找最近一個月的重要事件。
- [Significant Earthquakes｜重大地震](https://earthquake.usgs.gov/earthquakes/browse/significant.php)：依年份找事件。「重大」不只看規模，也考慮有感回報與可能影響。
- [Search Earthquake Catalog｜地震目錄查詢](https://earthquake.usgs.gov/earthquakes/search/)：設定時間、規模與區域，也可以選 CSV 輸出。

選好後，記下時間與震央位置，把下方網址改成事件前後幾天、震央附近的範圍，觀察主震周邊的地震分布；地圖的 `region` 也要配合調整。

**時間要對齊**：氣象署報告使用台灣時間（UTC+8），下方 USGS 查詢使用 UTC，要先減 8 小時，日期也可能變成前一天。這裡用氣象署找題目，實際繪圖資料仍來自 USGS；兩邊的規模與位置可能不同，不必強求完全一致。


### 用 USGS API 取得資料

向 USGS 要一份地震資料，再把經緯度畫到地圖上。API 就像點餐：在網址指定時間、區域與最低規模，USGS 就回傳符合條件的 CSV 表格。

這次查詢 **2020-01-01 至執行當下（UTC）、規模 ≥ 2**，範圍是東經 119–123 度、北緯 21–26 度，與前面的地圖相同。觀察地震位置與深度的空間變化；USGS 對台灣小地震的收錄不完整，這些點不代表所有台灣地震，也不是完整的隱沒板塊形狀。這組圖呈現多年地震分布；第一張另外標記 2024 花蓮主震來示範符號，不代表其他地震都是它的餘震。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/) | 按條件取得地震 CSV | `format`、`starttime`、`endtime`、`minmagnitude`、經緯度範圍 |
| [pd.read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) | 把 CSV 網址讀成表格 | `filepath_or_buffer`：此處傳入 `url` |

可以複製下格印出的網址到瀏覽器下載資料；每次執行都需要網路。


In [ ]:
import pandas as pd
from datetime import datetime, timezone


# 1. 設定結束時間：每次執行當下的 UTC 時間
endtime = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")


# 2. 組合查詢網址：修改下方時間、規模與範圍
url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query"  # USGS 地震查詢入口
    "?format=csv"                    # 回傳 CSV 表格；第一個參數用 ?
    "&starttime=2020-01-01"           # 開始時間（UTC）；後續參數用 &
    f"&endtime={endtime}"             # 結束時間（UTC）
    "&minmagnitude=2"                 # 最低地震規模
    "&minlongitude=119"               # 西界：最小經度
    "&maxlongitude=123"               # 東界：最大經度
    "&minlatitude=21"                 # 南界：最小緯度
    "&maxlatitude=26"                 # 北界：最大緯度
    "&orderby=time-asc"              # 依時間由早到晚排列
)


# 3. 讀取資料，去掉缺少繪圖欄位的地震
quakes = pd.read_csv(url)
quakes = quakes.dropna(subset=["longitude", "latitude", "mag", "depth"])


# 4. 確認查詢網址、筆數與前五筆資料
print(url)  # 可複製到瀏覽器，查看同一份 CSV
print(f"可繪製的地震：{len(quakes)} 筆；深度單位：km；時間：UTC。")

quakes[["time", "longitude", "latitude", "mag", "depth"]].head()


### 第一張：從 Pandas 表格取經緯度畫點
`quakes` 是 Pandas 的 DataFrame：每列是一筆地震，欄位包含經度、緯度、規模與深度。

這裡不是用 Pandas 畫圖，而是把 `quakes.longitude` 與 `quakes.latitude` 交給 PyGMT 的 `fig.plot()`。先讓所有點一樣大、同一種顏色，只看地震在哪裡。

**圓圈之外，也能畫星星**：`style="c0.12c"` 是直徑 0.12 cm 的圓圈，`style="a0.6c"` 是大小 0.6 cm 的星形。下格用星星標出 [2024 花蓮主震（USGS）](https://earthquake.usgs.gov/earthquakes/eventpage/us7000m9g4/executive)，示範直接指定經緯度。星星最後畫，會疊在圓圈上。

**試看看**：將星星的 `a` 改成 `t`（三角形）或 `s`（正方形），保持 `0.6c` 不變。每張圖都重新建立 `fig`，不會疊到上一張；第二、三張先不加這個標記。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [fig.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 畫點與不同符號 | `x`、`y`、`style`、`fill`、`pen`、`label` |
| [fig.legend()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.legend.html) | 顯示符號圖例 | `position`、`box` |


In [ ]:
import pygmt

fig = pygmt.Figure()
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame=["af", f"+tTaiwan 2020 to {endtime[:10]} | USGS"])
fig.coast(land="gray95", water="aliceblue", shorelines="0.5p,gray40", resolution="h")


# 從 Pandas 表格取出經緯度，每一列畫成一個點
fig.plot(
    x=quakes.longitude,  # 經度欄位：每筆地震的 x 位置
    y=quakes.latitude,  # 緯度欄位：每筆地震的 y 位置
    style="c0.12c",  # 所有圓圈的直徑都一樣：0.12 cm
    fill="tomato",  # 固定顏色：所有地震都用同一種色
    pen="0.25p,gray20",
    transparency=40,
)


# 單獨標出 2024 花蓮主震（USGS：us7000m9g4）
fig.plot(
    x=121.5976,       # 經度
    y=23.8356,        # 緯度
    style="a0.6c",    # a：星形；t：三角形；s：正方形
    fill="yellow",
    pen="1p,black",
    label="2024 Hualien mainshock",  # 新增：把星星與文字放進圖例
)

fig.legend(position="JTL+jTL+o0.2c", box="+gwhite+p0.5p")


fig.show()


### 第二張：讓地震規模決定大小
把固定大小的 `style="c0.12c"` 改成 `style="c"`，另外用 `size` 傳入每筆地震的圓圈直徑；顏色先維持不變。

用 `if / elif / else` 分區間，同一區間用固定直徑：M<3 → 0.035 cm、3≤M<4 → 0.07 cm、4≤M<5 → 0.14 cm、5≤M<6 → 0.28 cm、6≤M<7 → 0.56 cm、M≥7 → 1.12 cm。`plot_quakes.mag.apply(magnitude_size)` 會把每筆規模交給這個函式判斷。這是視覺設計，不是能量比例或影響半徑；規模也不是各地感受到的震度。

**試看看**：`ascending=True` 讓小圓先畫、大圓後畫；改成 `False`，比較重疊效果。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [fig.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 讓每筆地震有不同大小 | `size`、`style`、`transparency` |
| [DataFrame.sort_values()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) | 決定資料與繪圖順序 | `by`：此處為 `"mag"`；`ascending` |


In [ ]:
import pygmt

# 依規模區間指定圓圈直徑（cm）；同一區間使用相同大小
def magnitude_size(magnitude):
    if magnitude < 3:
        return 0.035
    elif magnitude < 4:
        return 0.07
    elif magnitude < 5:
        return 0.14
    elif magnitude < 6:
        return 0.28
    elif magnitude < 7:
        return 0.56
    else:
        return 1.12


# 小圓先畫，大圓後畫
plot_quakes = quakes.sort_values("mag", ascending=True)


fig = pygmt.Figure()
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame=["af", f"+tTaiwan 2020 to {endtime[:10]} | USGS"])
fig.coast(land="gray95", water="aliceblue", shorelines="0.5p,gray40", resolution="h")


# 改用 size，讓每筆地震有自己的圓圈大小
fig.plot(
    x=plot_quakes.longitude,
    y=plot_quakes.latitude,
    style="c",  # 改動：只指定圓形，大小改由下一行決定
    size=plot_quakes.mag.apply(magnitude_size),  # 新增：把每筆規模換成圓圈直徑
    fill="tomato",  # 沿用：這張先不改顏色
    pen="0.25p,gray20",
    transparency=40,
)


fig.show()


### 第三張：用深度上色，認識 CPT

前一張用規模控制大小；這張保留大小，再用 `fill=plot_quakes.depth` 與 `cmap=True` 把深度轉成顏色，最後加上 color bar。

CPT（Color Palette Table）決定數值對應什麼顏色。從下方色票總覽挑一組喜歡的配色，把名稱填進 `pygmt.makecpt(cmap=...)`。

- 這次用 `cmap="gmt/seis"`：隨深度增加，由紅、橙、黃、綠轉為藍，也就是淺層紅、深層藍。
- `gmt/` 是色票分類，`seis` 是名稱；也可以試 `jet` 或 `rainbow`，比較同一份資料的呈現。
- `series=[0, 150, 1]` 將色票固定在 0–150 km；`background=True` 讓超過 150 km 的地震沿用最深端顏色，不會刪除這些事件。`reverse=True` 可以反轉顏色順序。

**試看看**：只改 `cmap`，重跑下格，比較哪些深度比較醒目。深度要對照 color bar，不能只憑明暗判斷；目前圓圈有 40% 透明度，顏色也會受底圖和重疊影響。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [GMT 色票總覽](https://docs.generic-mapping-tools.org/6.5/reference/cpts.html) | 比較內建 CPT（色票圖鑑，非函式） | 色票名稱，如 `gmt/seis`、`jet` |
| [pygmt.makecpt()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.makecpt.html) | 建立數值到顏色的對應 | `cmap`、`series`、`reverse` |
| [fig.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 依地震深度上色 | `fill`、`cmap=True` |
| [fig.colorbar()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.colorbar.html) | 顯示色條與單位 | `frame` |


In [ ]:
import pygmt

from io import StringIO


# 沿用上一格的 magnitude_size 與 plot_quakes

# 1. 建立新圖與深度色票
# 深度色票固定為 0–150 km，方便不同圖之間比較

fig = pygmt.Figure()

# gmt/seis：淺層紅，經橙、黃、綠，轉為深層藍
pygmt.makecpt(
    cmap="gmt/seis",  # 新增：選擇色票名稱
    series=[0, 150, 1],  # 深度起點、終點、間隔（km）；上限固定 150
    background=True,  # 超過 150 km 沿用色票最深端的顏色
    continuous=True,  # 新增：建立連續漸層
)


# 2. 畫底圖與海岸線
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame=["af", f"+tTaiwan 2020 to {endtime[:10]} | USGS"])
fig.coast(land="gray95", water="aliceblue", shorelines="0.5p,gray40", resolution="h")


# 用深度控制填色，規模繼續控制大小
fig.plot(
    x=plot_quakes.longitude,
    y=plot_quakes.latitude,
    size=plot_quakes.mag.apply(magnitude_size),  # 沿用：規模仍控制大小
    style="c",
    fill=plot_quakes.depth,  # 改動：不再填固定色名，改傳入每筆深度
    cmap=True,  # 新增：用剛建立的 CPT 把深度數值轉成顏色
    pen="0.25p,gray20",
    transparency=40,  # 0 不透明，100 完全透明
)


# 3. 加上規模圖例（留出大圓需要的間距）
legend = StringIO(
    "".join(
        f"S 0.6c c {magnitude_size(mag):.3f}c gray70 0.25p,gray20 1.4c {label}\n"
        f"G {max(0.15, magnitude_size(mag) - 0.25):.2f}c\n"
        for mag, label in [
            (2, "M < 3"),
            (3, "3 <= M < 4"),
            (4, "4 <= M < 5"),
            (5, "5 <= M < 6"),
            (6, "6 <= M < 7"),
            (7, "M >= 7"),
        ]
    )
)

fig.legend(
    spec=legend,
    position="JTL+jTL+o0.2c",
    box="+gwhite+p0.5p",
)


# 4. 加上深度色條
fig.colorbar(
    frame=["xaf", "y+lDepth (km)"],  # 新增：顯示色票對應的刻度與深度單位
)


# 5. 顯示圖片
fig.show()


## 3. 彩色地形：高度變成顏色
`load_earth_relief()` 取得地形網格，`grdimage()` 把高度畫成顏色。`02m` 指 2 角分，**不是 2 公尺**。第一次執行需要下載 GMT 地形資料。

**試看看**：把 `shading=True` 改成 `False`，比較起伏的可讀性。深色也可能來自陰影，不能只用明暗判斷高低。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [load_earth_relief()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.datasets.load_earth_relief.html) | 載入地形與海底高程網格 | `resolution`、`region`、`registration` |
| [fig.grdimage()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdimage.html) | 將網格畫成彩色地形 | `grid`、`cmap`、`shading` |
| [fig.colorbar()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.colorbar.html) | 顯示高程色條與單位 | `frame` |


In [ ]:
import pygmt

# 1. 取得地形網格
grid = pygmt.datasets.load_earth_relief(
    resolution="02m",  # 新增：網格間距 2 角分，不是 2 公尺
    region=[119, 123, 21, 26],  # 下載範圍：西、東、南、北
    registration="gridline",  # 網格值位於格線交點，這裡先沿用
)

print("地形網格：", grid.shape, "；高程單位：m")


# 2. 用高程上色，不再逐筆畫地震點
fig = pygmt.Figure()

fig.grdimage(
    grid=grid,  # 新增：輸入整片地形網格
    region=[119, 123, 21, 26],
    projection="M15c",
    cmap="geo",  # 改動：使用海底與陸地的地形色票
    shading=True,  # 新增：加入陰影，凸顯起伏
    frame=["af", "+tTaiwan: land and seafloor"],
)


# 3. 疊上海岸線與高程色條
fig.coast(shorelines="0.5p,gray25", resolution="h")
fig.colorbar(frame=["xaf", "y+lElevation (m)"])  # 改動：現在表示高程，單位 m


fig.show()


## 4. 3D 地形：換個角度看台灣
用 `grdview()` 將同一份地形畫成斜視圖。`perspective=[方位角, 仰角]` 控制觀看方向，`zsize` 控制垂直尺寸。

圖中垂直方向為了辨認起伏而誇大，不能當成真實坡度。這是固定視角圖片，不是滑鼠可拖曳的模型。

**試看看**：把方位角 135 改為 225，仰角維持 35，觀察哪些山被遮住。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [fig.grdview()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdview.html) | 繪製斜視地形表面 | `grid`、`region`、`perspective`、`zsize`、`surftype` |


In [ ]:
import pygmt


# 1. 設定觀看方向
azimuth = 135  # 方位角：繞地形從哪個方向看（度）
elevation = 35  # 仰角：觀看角度的高低（度）


# 2. 沿用上一格的 grid，改畫 3D 地形
fig = pygmt.Figure()

fig.grdview(
    grid=grid,  # 沿用：同一份地形資料
    region=[119, 123, 21, 26, -8000, 4000],  # 新增：最後兩個值是高程下、上限（m）
    projection="M15c",
    perspective=[azimuth, elevation],  # 新增：方位角、仰角
    zsize="3c",  # 新增：垂直軸畫成多高；不是實際山高
    surftype="s",  # 新增：繪製表面
    cmap="geo",  # 沿用：高程色票
    frame=[
        "xaf",  # x 軸刻度
        "yaf",  # y 軸刻度
        "zaf+lElevation (m)",  # 新增：垂直軸刻度與單位
        "+tTaiwan: 3D relief",
    ],
)


# 3. 顯示圖片
fig.show()


### AI 前導：把 3D 地形接上旋轉拉桿
剛才我們直接修改程式裡的角度；現在看看 AI 可以怎麼幫忙，把同樣的參數接上拉桿，讓我們更方便探索視角。完成環境安裝後可直接執行下格；本格會自行載入地形，不依賴其他 cell。

- **方位角**：繞地形旋轉，看看台灣的不同側面。
- **仰角**：由低角度斜看，逐漸拉高到接近俯視。

放開拉桿後才重新繪圖，避免拖動時連續計算。這是互動更新的靜態圖片，不是即時 3D 模型；需要運作中的 Colab／Jupyter，GitHub 靜態預覽不能操作。

**請 AI 延伸**：加上垂直高度 `zsize` 的拉桿，或「回到預設視角」按鈕。若想做時間動畫，可以改用前面的地震資料，逐月更新地震點，並固定色票與座標範圍。

提示詞：
> 請在這個 PyGMT＋ipywidgets 範例加上垂直高度拉桿。放開拉桿才更新，沿用已載入的 grid，不要每次重新下載。說明哪個控制項對應哪個繪圖參數。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [widgets.interact()](https://ipywidgets.readthedocs.io/en/stable/examples/Using%20Interact.html) | 將拉桿連到繪圖函式 | 函式參數與控制項的對應 |
| [IntSlider](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html#IntSlider) | 建立整數拉桿 | `min`、`max`、`step`、`continuous_update` |
| [fig.grdview()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdview.html) | 依拉桿值重畫地形 | `perspective`、`zsize` |


In [ ]:
import pygmt
import ipywidgets as widgets


# 1. 本格自行載入地形；拉動滑桿時不重新下載
grid = pygmt.datasets.load_earth_relief(
    resolution="02m",
    region=[119, 123, 21, 26],
    registration="gridline",
)


# 2. 把原本的 3D 繪圖包成函式，讓拉桿傳入角度
def rotate_terrain(azimuth, elevation):
    fig = pygmt.Figure()

    fig.grdview(
        grid=grid,  # 沿用：已下載的地形，不重複下載
        region=[119, 123, 21, 26, -8000, 4000],
        projection="M12c",
        perspective=[azimuth, elevation],  # 改動：由拉桿決定視角
        zsize="2.4c",
        surftype="s",
        cmap="geo",
        frame=["xaf", "yaf", "zaf", "+tTaiwan relief"],
    )

    fig.show(dpi=100)  # 降低預覽解析度，加快更新


# 3. 每個拉桿對應函式的一個參數；放開滑鼠後才更新
widgets.interact(
    rotate_terrain,
    azimuth=widgets.IntSlider(
        value=135,
        min=0,
        max=360,
        step=5,
        description="方位角",
        continuous_update=False,
    ),
    elevation=widgets.IntSlider(
        value=35,
        min=10,
        max=85,
        step=5,
        description="仰角",
        continuous_update=False,
    ),
);


## 5. 開始用 AI：先想問題，再決定怎麼畫

不用提出很創新或複雜的研究。先找一份資料，想一個你想觀察的小問題，再和 AI 討論怎麼篩選、排序、比較與呈現。

1. **先寫一句想法**：我想看看什麼？例如「把台灣地震依深度上色，看看是否呈現與隱沒帶相符的空間排列」。這是待探索的想法，不是預先確定的結論。
2. **想想資料怎麼整理**：要選哪個區域、時間或深度？換個排序、分組或剖面方向，會不會看出原本沒注意到的特徵？
3. **和 AI 討論畫法**：地圖、剖面或其他圖，哪種更能回答問題？哪些部分用 PyGMT 完成？也可以搭配其他資料處理或互動工具，不必只用 PyGMT。
4. **看圖，再修正說法**：圖能不能讓人一眼抓到重點？若沒有看到預期特徵，也可以如實呈現；不要為了符合猜想而只挑支持它的資料。

最後把「你想表達的文字」和「圖」放在一起：先用一句話引導作圖，完成後再改成符合實際結果的簡短圖說，交代想看什麼、圖上怎麼呈現、實際看到什麼。簡單幾句就好，不用寫成研究報告，也不用證明假設成立。

可以這樣開始和 AI 討論：

> 我有＿＿資料，想看看＿＿。請先和我討論要怎麼篩選、排序或比較，以及哪種圖最容易看出重點，再寫程式。作品要用到 PyGMT，但可搭配其他工具。請幫我檢查圖例、尺度和資料限制；若結果不支持原本的想法，也要保留並說明。

剛才的旋轉拉桿與下方的 A–B 剖面都是工具靈感，不是指定作業形式。


### 先逛逛論文的圖，找找靈感

不用一開始就讀懂整篇 paper。先到 [Google Scholar](https://scholar.google.com/) 搜尋 `seismic`、`earthquake` 或 `seismicity`，也可以加上 `Taiwan`、`subduction`、`cross section`、`waveform` 等地區或圖像關鍵字。或先用 Google 圖片搜尋，看到有興趣的圖，再回到原論文看圖說。

也可以直接逛這些期刊，挑一篇題目有興趣的文章，先翻圖片：

| 期刊入口 | 主要範圍 | 逛圖時可以找什麼 |
| --- | --- | --- |
| [SRL — Seismological Research Letters](https://pubs.geoscienceworld.org/srl) | 地震學及相關觀測、方法與應用 | 地震事件、測站、波形與資料展示。 |
| [BSSA — Bulletin of the Seismological Society of America](https://pubs.geoscienceworld.org/bssa) | 地震學與相關研究 | 地震分布、震源、地動與分析結果。 |
| [GJI — Geophysical Journal International](https://academic.oup.com/gji) | 固體地球物理，不限地震 | 地下構造、剖面、波形與模型比較。 |
| [GRL — Geophysical Research Letters](https://agupubs.onlinelibrary.wiley.com/journal/19448007) | 地球與太空科學，不限地震 | 搜尋地震相關文章，看作者怎麼用少量圖呈現重點。 |
| [Seismica](https://seismica.library.mcgill.ca/) | 地震學與地震科學，開放取用 | 地震研究、資料與方法的各種呈現方式。 |

挑一張喜歡的圖就好，想想：**它想表達什麼？資料怎麼篩選或排列？我可以借用哪種畫法來表達自己的問題？** 重點是學呈現方式，不是照抄結論，也不必做出同樣複雜的研究。遇到付費文章，可找開放版本或換一篇。

把原論文連結與圖號留給自己，也可以給 AI 當討論參考。圖片搜尋只是入口，仍要回原文確認圖說；若要把原圖放進公開 GitHub，需確認授權並標明來源。



### 資料也可以換，找找新的靈感

地震資料不只有 USGS；先想清楚要的是「地震發生在哪裡」的目錄，還是「測站記錄到怎麼搖」的波形。

| 想找什麼 | 資料入口 | 可以做什麼 |
| --- | --- | --- |
| 台灣更細的地震資料 | [氣象署 GDMS](https://gdms.cwa.gov.tw/) | 查找台灣地震目錄與波形；想研究小地震或局部構造，可以從這裡找起，下載方式與權限依網站說明。 |
| 全球地震目錄 | [USGS](https://earthquake.usgs.gov/fdsnws/event/1/)／[ISC Bulletin](https://www.isc.ac.uk/iscbulletin/search/) | 取得時間、位置、深度與規模，畫分布圖或剖面；各目錄的收錄範圍與更新速度不同。 |
| 全球測站的地震波形 | [EarthScope（原 IRIS 服務）](https://service.earthscope.org/fdsnws/dataselect/1/) | 按測站、通道與時間下載波形，試做「震央與測站地圖＋波形」；不是每個測站都有所有時段的資料。 |

波形不是地震目錄，不能直接套進本課的地震點位程式。可以請 AI 協助讀取、處理，再用 PyGMT 呈現；例如畫出同一場地震在不同測站的記錄。以上只是靈感，不是額外作業要求。


### AI 展示：在地圖點 A、B，切開地震分布
完成環境安裝後，本格可獨立執行，會自行下載 2020 年至今的 USGS 地震。

1. 在互動地圖點一下設定 **A**，再點一下設定 **B**。
2. 用「半寬 km」拉桿決定剖面線兩側要選多寬；藍色範圍就是取樣走廊。
3. 按「更新剖面」，先看標有 A、B、剖面線與取樣走廊的 PyGMT 地圖，再看由 A 往 B 的距離—深度圖。換位置前按「重選 A、B」。

可以先試 A 約在花蓮南方、B 在台灣東北方；不必限制南北向。地圖用 ipyleaflet 操作，PyGMT 負責地圖與剖面出圖，`project()` 計算沿線距離與離線距離（球面近似）。

**讀圖注意**：深度向下增加；超過 150 km 的事件沿用最深端藍色。水平與垂直比例不一定相同，不能直接量圖上的傾角。已下載的資料只涵蓋 119–123°E、21–26°N；在框外畫剖面不會自動取得新地震。地震排列也不是完整板塊邊界。

互動地圖需網路與運作中的 Colab／Jupyter；GitHub 靜態預覽無法點選。若 Colab 提示允許自訂元件，請確認後啟用。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [ipyleaflet Map](https://ipyleaflet.readthedocs.io/en/latest/map_and_basemaps/map.html) | 互動地圖與點擊事件 | `on_interaction`、`coordinates` |
| [pygmt.project()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.project.html) | 投影及篩選剖面地震 | `center`、`endpoint`、`width`、`length`、`unit` |
| [Jupyter Widgets](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html) | 拉桿與按鈕 | `IntSlider`、`Button`、`Output` |
| [Figure.coast()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.coast.html) | 地圖底圖 | `region`、`projection`、`land` |
| [Figure.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 地震、剖面線與走廊 | `x`、`y`、`close`、`pen` |
| [Figure.text()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.text.html) | A、B 標籤 | `text`、`offset`、`font` |


In [ ]:
import pygmt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipyleaflet import Map, Marker, Polyline, Polygon, CircleMarker, LayerGroup
from IPython.display import display, clear_output
from datetime import datetime, timezone
from pyproj import Geod

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass  # 本機 Jupyter 不需要 Colab 的元件管理器


# 1. 本格自行下載資料，不依賴其他 cell
endtime = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
    "&starttime=2020-01-01"
    f"&endtime={endtime}"
    "&minmagnitude=2"
    "&minlongitude=119&maxlongitude=123"
    "&minlatitude=21&maxlatitude=26"
)
quakes = pd.read_csv(url).dropna(subset=["longitude", "latitude", "depth", "mag"])

def magnitude_size(magnitude):
    if magnitude < 3:
        return 0.035
    elif magnitude < 4:
        return 0.07
    elif magnitude < 5:
        return 0.14
    elif magnitude < 6:
        return 0.28
    elif magnitude < 7:
        return 0.56
    else:
        return 1.12


# 2. 地圖只負責點選；GMT project 計算沿線距離與垂直距離
# 使用球面近似，走廊邊界也用同一種球面幾何
sphere = Geod(a=6371008.8, f=0)
points = []  # 點擊順序：A、B；每個座標為（緯度、經度）
map_view = Map(center=(24, 121.5), zoom=7, scroll_wheel_zoom=True,
               layout=widgets.Layout(height="450px"))
earthquake_layer = LayerGroup(layers=tuple(
    CircleMarker(location=(row.latitude, row.longitude), radius=2,
                 color="#777777", fill_color="#777777", fill_opacity=0.4, weight=0)
    for row in quakes.itertuples()
))
selection_layer = LayerGroup()
map_view.add(earthquake_layer)
map_view.add(selection_layer)

width_slider = widgets.IntSlider(value=40, min=10, max=100, step=5,
    description="半寬 km", continuous_update=False)
update_button = widgets.Button(description="更新剖面", button_style="primary")
reset_button = widgets.Button(description="重選 A、B")
status = widgets.HTML("請在地圖點 A，再點 B。灰點為已下載的地震。")
output = widgets.Output()


def profile_geometry():
    (lat_a, lon_a), (lat_b, lon_b) = points
    azimuth, _, meters = sphere.inv(lon_a, lat_a, lon_b, lat_b)
    if meters < 1000:
        raise ValueError("A、B 請至少相距 1 km。")
    along = np.linspace(0, meters, 60)
    lons, lats, back = sphere.fwd(
        np.full(60, lon_a), np.full(60, lat_a), np.full(60, azimuth), along
    )
    # 軌跡每個位置的前進方向，左右各延伸半寬
    left_lon, left_lat, _ = sphere.fwd(lons, lats, back + 90, np.full(60, width_slider.value * 1000))
    right_lon, right_lat, _ = sphere.fwd(lons, lats, back - 90, np.full(60, width_slider.value * 1000))
    track = list(zip(lats, lons))
    corridor = list(zip(left_lat, left_lon)) + list(zip(right_lat, right_lon))[::-1]
    return meters / 1000, track, corridor


# 3. 點選或改寬度時，更新地圖範圍並清掉舊剖面
def refresh_selection():
    with output:
        clear_output(wait=False)
    layers = [Marker(location=p, title=label, draggable=False) for p, label in zip(points, ["A", "B"])]
    if len(points) == 2:
        try:
            length, track, corridor = profile_geometry()
        except ValueError as error:
            status.value = str(error)
        else:
            layers += [Polygon(locations=corridor, color="#1565c0", fill_opacity=0.12),
                       Polyline(locations=track, color="#1565c0", weight=3)]
            status.value = f"A={points[0]}；B={points[1]}；長約 {length:.0f} km，全寬 {2*width_slider.value} km。請按更新剖面。"
    else:
        status.value = "請點 B。" if points else "請點 A，再點 B。"
    selection_layer.layers = tuple(layers)

def on_map_click(**event):
    if event.get("type") != "click":
        return
    if len(points) == 2:
        status.value = "要換位置，請先按「重選 A、B」。"
        return
    points.append(tuple(event["coordinates"]))
    refresh_selection()

def reset_selection(_):
    points.clear()
    refresh_selection()


# 4. 按鈕才觸發計算與 GMT 出圖；深度向下增加
def draw_section(_=None):
    with output:
        clear_output(wait=True)
        if len(points) != 2:
            print("請先在地圖選好 A、B。")
            return
        try:
            length, track, corridor = profile_geometry()
        except ValueError as error:
            print(error)
            return

        selected = pygmt.project(
            data=quakes[["longitude", "latitude", "depth", "mag"]],
            center=list(points[0][::-1]),  # GMT 順序：經度、緯度
            endpoint=list(points[1][::-1]),
            unit=True,  # 距離單位 km
            length="w",  # 只保留 A 到 B 之間
            width=[-width_slider.value, width_slider.value],  # 線兩側的半寬
            convention="xypqz",  # 經度、緯度、沿線距離、離線距離、深度、規模
        )
        if selected.empty:
            print("範圍內沒有地震，請改位置或加大半寬。")
            return
        selected.columns = ["longitude", "latitude", "distance", "offset", "depth", "mag"]
        selected = selected.sort_values("mag")
        print(f"取樣 {len(selected)} / {len(quakes)} 筆；2020-01-01 至 {endtime} UTC")
        print("只含下載範圍：119–123°E、21–26°N；走廊超出此範圍的部分沒有資料。")

        # 新增：先畫地理位置圖，灰點為全部地震，彩色點為剖面取樣
        map_fig = pygmt.Figure()
        pygmt.makecpt(cmap="gmt/seis", series=[0, 150, 1], continuous=True, background=True)
        corridor_lat, corridor_lon = np.array(corridor).T
        track_lat, track_lon = np.array(track).T
        map_fig.coast(
            region=[min(119, corridor_lon.min()-0.1), max(123, corridor_lon.max()+0.1),
                    min(21, corridor_lat.min()-0.1), max(26, corridor_lat.max()+0.1)],
            projection="M14c",
            land="gray95", water="aliceblue", shorelines="0.5p,gray40",
            frame=["af", "+tA-B profile location"],
        )
        map_fig.plot(x=quakes.longitude, y=quakes.latitude, style="c0.035c", fill="gray70")
        map_fig.plot(
            x=selected.longitude, y=selected.latitude,
            style="c", size=selected.mag.apply(magnitude_size),
            fill=selected.depth, cmap=True, transparency=40, pen="0.2p,gray30",
        )

        # 走廊只畫外框，避免蓋住地震的深度顏色
        map_fig.plot(x=corridor_lon, y=corridor_lat, close=True, pen="1p,blue,--")
        map_fig.plot(x=track_lon, y=track_lat, pen="1.5p,black")
        map_fig.plot(x=[points[0][1], points[1][1]], y=[points[0][0], points[1][0]],
                     style="s0.22c", fill="white", pen="1p,black")
        map_fig.text(
            x=[points[0][1], points[1][1]], y=[points[0][0], points[1][0]],
            text=["A", "B"], font="14p,Helvetica-Bold,black",
            offset="0.2c/0.2c", justify="BL", fill="white",
        )
        map_fig.colorbar(frame=["xaf", "y+lDepth (km)"])
        map_fig.show()

        # 再畫剖面：沿用同一套規模大小與 0–150 km 深度色階
        fig = pygmt.Figure()
        pygmt.makecpt(cmap="gmt/seis", series=[0, 150, 1], continuous=True, background=True)
        fig.basemap(
            region=[0, length, min(0, np.floor(selected.depth.min()/50)*50), max(50, np.ceil(selected.depth.max()/50)*50)],
            projection="X14c/-9c",
            frame=["xaf+lDistance from A (km)", "yaf+lDepth (km)", "+tA to B"],
        )
        fig.plot(
            x=selected.distance,
            y=selected.depth,
            style="c",
            size=selected.mag.apply(magnitude_size),
            fill=selected.depth,
            cmap=True,
            transparency=40,
            pen="0.2p,gray30",
        )
        fig.colorbar(frame=["xaf", "y+lDepth (km)"])
        fig.show()


map_view.on_interaction(on_map_click)
width_slider.observe(lambda change: refresh_selection(), names="value")
reset_button.on_click(reset_selection)
update_button.on_click(draw_section)
display(widgets.VBox([map_view, widgets.HBox([width_slider, update_button, reset_button]), status, output]))


## 6. 隔週繳交
這次作業就用 AI 做！想想你想呈現什麼，讓 AI 幫你把點子做出來。還沒靈感的話，可以先逛逛 [PyGMT Gallery](https://www.pygmt.org/v0.17.0/gallery/index.html)，找喜歡的範例，再試著改造或組合。

作品請把**圖＋一小段圖說**放在一起：說明你想看什麼、如何呈現，以及實際觀察到什麼。不必創新或複雜，也不必得到符合原先猜想的結果；重點是讓人一眼抓到你想表達的事。可以搭配其他工具，不必只用 PyGMT。

作業只需滿足兩個條件：

1. **作品與 PyGMT 有關。**
2. **將作品上傳 GitHub，繳交 repository 連結**，並確認教師能開啟。

題材、區域、呈現形式與圖的張數都可自由發揮，不限定沿用課堂範例；隔週繳交即可。


## 資料來源與版本

- [原始課程參考 Notebook](https://github.com/oceanicdayi/plot_plate_boundary_pygmt/blob/main/pygmt_plot_plate_boundary.ipynb)
- [GMT 全球地形資料](https://docs.generic-mapping-tools.org/latest/datasets/remote-data.html)：PyGMT 載入，首次使用需連網。
- [PyGMT 0.17 安裝文件](https://www.pygmt.org/v0.17.0/install.html)


| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/) | 本課使用的真實事件資料 | 查詢條件包含在下載網址中 |

